<a href="https://colab.research.google.com/github/Nanokwok/Deep-Fraud-VAE/blob/main/compare/model_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Comparison: β-VAE vs Baselines
**Credit Card Fraud Detection — Kaggle Dataset**

Compares five approaches on the same test set (held out from all models):

| Model | Paradigm | Trains on fraud labels? |
|---|---|---|
| β-VAE (ours) | Semi-supervised anomaly detection | No |
| Isolation Forest | Unsupervised anomaly detection | No |
| One-Class SVM | Semi-supervised anomaly detection | No |
| Logistic Regression | Supervised | Yes |
| Random Forest | Supervised | Yes |
| XGBoost | Supervised | Yes |


## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/nanokwok/Deep-Fraud-VAE.git"
    REPO_DIR = "/content/Deep-Fraud-VAE"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    sys.path.insert(0, REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"], check=True)
    ROOT_DIR = REPO_DIR
else:
    _here = Path().resolve()
    ROOT_DIR = str(_here.parent if _here.name == "compare" else _here)
    os.chdir(ROOT_DIR)
    sys.path.insert(0, ROOT_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "xgboost", "-q"], check=True)

COMPARE_DIR = os.path.join(ROOT_DIR, "compare")
os.makedirs(COMPARE_DIR, exist_ok=True)

print(f"ROOT_DIR    : {ROOT_DIR}")
print(f"Environment : {'Colab' if IN_COLAB else 'Local'}")


## 2. Data

The notebook tries three sources in order:
1. **Google Drive** — if you already ran the training notebook, the `.npy` files are already there. No download needed.
2. **Repo `data/processed/`** — if running locally with preprocessed data.
3. **Kaggle download** — only if neither above is available. Fill in your credentials before running.


In [ ]:
import json as _json
from pathlib import Path

# ── Source priority ────────────────────────────────────────────────────────
DRIVE_PROC = Path("/content/drive/MyDrive/Fraud_VAE_Project/data/processed")
REPO_PROC  = Path(ROOT_DIR) / "data" / "processed"

if IN_COLAB and DRIVE_PROC.exists() and (DRIVE_PROC / "X_train.npy").exists():
    proc_dir = DRIVE_PROC
    print(f"Loading from Google Drive: {proc_dir}")

elif (REPO_PROC / "X_train.npy").exists():
    proc_dir = REPO_PROC
    print(f"Loading from repo: {proc_dir}")

else:
    # ── Kaggle download (fill in credentials if needed) ───────────────────
    if IN_COLAB:
        KAGGLE_USERNAME = "your_kaggle_username"   # ← replace
        KAGGLE_KEY      = "your_kaggle_api_key"    # ← replace

        kaggle_dir = os.path.expanduser("~/.kaggle")
        os.makedirs(kaggle_dir, exist_ok=True)
        with open(f"{kaggle_dir}/kaggle.json", "w") as _f:
            _json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, _f)
        os.chmod(f"{kaggle_dir}/kaggle.json", 0o600)

        raw_dir = os.path.join(ROOT_DIR, "data", "raw")
        raw_csv = os.path.join(raw_dir, "creditcard.csv")
        os.makedirs(raw_dir, exist_ok=True)

        if not os.path.exists(raw_csv):
            print("Downloading Credit Card Fraud dataset from Kaggle (~144 MB)...")
            subprocess.run([
                "kaggle", "datasets", "download",
                "mlg-ulb/creditcardfraud",
                "-p", raw_dir, "--unzip",
            ], check=True)
            print(f"Download complete: {os.path.getsize(raw_csv)/1e6:.1f} MB")

    print("Running preprocessing...")
    from src.preprocess import preprocess
    preprocess()
    proc_dir = REPO_PROC

# ── Load arrays ────────────────────────────────────────────────────────────
X_train = np.load(proc_dir / "X_train.npy").astype(np.float32)
X_val   = np.load(proc_dir / "X_val.npy").astype(np.float32)
X_test  = np.load(proc_dir / "X_test.npy").astype(np.float32)
y_train = np.load(proc_dir / "y_train.npy").astype(np.int32)
y_val   = np.load(proc_dir / "y_val.npy").astype(np.int32)
y_test  = np.load(proc_dir / "y_test.npy").astype(np.int32)

print(f"X_train : {X_train.shape}  — normal only")
print(f"X_val   : {X_val.shape}  fraud_rate={y_val.mean()*100:.2f}%")
print(f"X_test  : {X_test.shape}  fraud_rate={y_test.mean()*100:.2f}%  (evaluation set)")


## 3. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_recall_curve
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps"  if torch.backends.mps.is_available() else "cpu"
)
print(f"Device: {device}")


## 4. Build Supervised Training Set

Supervised models (LR, RF, XGBoost) train on `X_train` (all normal) combined with `X_val` (normal + fraud), and are tested on `X_test`.
Semi-supervised models (VAE, IF, OCSVM) train on `X_train` (normal only).


In [ ]:
X_sup = np.vstack([X_train, X_val])
y_sup = np.concatenate([y_train, y_val])

fraud_count  = int(y_sup.sum())
normal_count = int((y_sup == 0).sum())
print(f"Supervised train  normal={normal_count:,}  fraud={fraud_count:,}  "
      f"imbalance_ratio={normal_count/fraud_count:.0f}:1")


## 5. Evaluation Helper

In [ ]:
results: dict = {}
pr_curves: dict = {}

def evaluate_model(name: str, y_true: np.ndarray, scores: np.ndarray) -> dict:
    """Compute AUPRC, AUROC, and best-F1 threshold metrics."""
    auprc = average_precision_score(y_true, scores)
    auroc = roc_auc_score(y_true, scores)

    prec, rec, thresholds = precision_recall_curve(y_true, scores)
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    best_idx = int(np.argmax(f1s))
    best_t   = float(thresholds[best_idx])

    y_pred = (scores >= best_t).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    p  = tp / (tp + fp + 1e-9)
    r  = tp / (tp + fn + 1e-9)
    f1 = 2 * p * r / (p + r + 1e-9)

    row = dict(
        Model=name, AUPRC=round(auprc,4), AUROC=round(auroc,4),
        Precision=round(p,4), Recall=round(r,4), F1=round(f1,4),
    )
    results[name]   = row
    pr_curves[name] = (prec, rec)

    print(f"[{name:<22}]  AUPRC={auprc:.4f}  AUROC={auroc:.4f}  "
          f"F1={f1:.4f}  P={p:.4f}  R={r:.4f}")
    return row


## 6. β-VAE (Ours)

Uses the pretrained checkpoint committed to the repo (`models/best_model.pth`). **No retraining** — just a forward pass (~seconds) to compute weighted reconstruction errors on the test set. Same model weights as evaluated in `04_evaluate.ipynb`.


In [ ]:
from src.model import BetaVAE
import src.config as cfg

CKPT_PATH = os.path.join(ROOT_DIR, "models", "best_model.pth")

# Build feature weight vector (same as training)
_FEATURE_COLS = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount"]
feat_w = torch.ones(cfg.INPUT_DIM, dtype=torch.float32, device=device)
for feat, w in cfg.FEATURE_WEIGHTS.items():
    if feat in _FEATURE_COLS:
        feat_w[_FEATURE_COLS.index(feat)] = w

# Load checkpoint
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=True)
input_dim  = ckpt.get("input_dim",  cfg.INPUT_DIM)
latent_dim = ckpt.get("latent_dim", cfg.LATENT_DIM)
_orig = cfg.LATENT_DIM
cfg.LATENT_DIM = latent_dim
vae = BetaVAE(input_dim=input_dim).to(device)
cfg.LATENT_DIM = _orig
vae.load_state_dict(ckpt["model_state"])
vae.eval()

# Score test set
x_t = torch.from_numpy(X_test).to(device)
with torch.no_grad():
    x_hat, _, _ = vae(x_t)
    sq = (x_t - x_hat) ** 2 * feat_w.unsqueeze(0)
    vae_scores = sq.mean(dim=1).cpu().numpy()

evaluate_model("Beta-VAE", y_test, vae_scores)


## 7. Isolation Forest

Unsupervised. Trained on `X_train` (normal only) like the VAE. Anomaly score = negated `decision_function` (higher = more anomalous).


In [ ]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(n_estimators=100, contamination="auto", random_state=42, n_jobs=-1)
iso.fit(X_train)
iso_scores = -iso.decision_function(X_test)

evaluate_model("Isolation Forest", y_test, iso_scores)


## 8. One-Class SVM

Semi-supervised. Trained on a subsample of `X_train` (normal only) — OCSVM is O(n²) in memory so we cap at 10,000 samples for speed.


In [ ]:
from sklearn.svm import OneClassSVM

rng = np.random.default_rng(42)
n_ocsvm = min(10_000, len(X_train))
idx = rng.choice(len(X_train), n_ocsvm, replace=False)

print(f"Training OCSVM on {n_ocsvm:,} samples (subsample of {len(X_train):,})...")
ocsvm = OneClassSVM(kernel="rbf", nu=0.01)
ocsvm.fit(X_train[idx])
ocsvm_scores = -ocsvm.decision_function(X_test)

evaluate_model("One-Class SVM", y_test, ocsvm_scores)


## 9. Logistic Regression

Supervised baseline. `class_weight='balanced'` compensates for extreme imbalance.


In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_sup, y_sup)
lr_scores = lr.predict_proba(X_test)[:, 1]

evaluate_model("Logistic Regression", y_test, lr_scores)


## 10. Random Forest

Supervised. `class_weight='balanced'` reweights samples per tree.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1
)
rf.fit(X_sup, y_sup)
rf_scores = rf.predict_proba(X_test)[:, 1]

evaluate_model("Random Forest", y_test, rf_scores)


## 11. XGBoost

Supervised. `scale_pos_weight = normal_count / fraud_count` balances gradients.


In [ ]:
import xgboost as xgb

xgb_clf = xgb.XGBClassifier(
    n_estimators=200,
    scale_pos_weight=normal_count / fraud_count,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
xgb_clf.fit(X_sup, y_sup)
xgb_scores = xgb_clf.predict_proba(X_test)[:, 1]

evaluate_model("XGBoost", y_test, xgb_scores)


## 12. Summary

In [ ]:
df = (
    pd.DataFrame(results.values())
    .set_index("Model")
    .sort_values("AUPRC", ascending=False)
)

display(
    df.style
    .highlight_max(axis=0, props="font-weight: bold; color: #2ecc71")
    .highlight_min(axis=0, props="color: #e74c3c")
    .format("{:.4f}")
    .set_caption("Model Comparison — Test Set (sorted by AUPRC)")
)


## 13. Precision-Recall Curve Comparison

In [ ]:
colors = ["#e74c3c", "#3498db", "#9b59b6", "#2ecc71", "#f39c12", "#1abc9c"]

fig, ax = plt.subplots(figsize=(9, 6))

for (name, (prec, rec)), color in zip(pr_curves.items(), colors):
    auprc = results[name]["AUPRC"]
    ax.plot(rec, prec, label=f"{name}  (AUPRC={auprc:.4f})", color=color, linewidth=2)

baseline = y_test.mean()
ax.axhline(baseline, color="grey", linestyle="--", linewidth=1,
           label=f"Random baseline (AUPRC={baseline:.4f})")

ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title("Precision-Recall Curve — Credit Card Fraud Detection", fontsize=13)
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.legend(fontsize=9, loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()

out_path = os.path.join(COMPARE_DIR, "pr_curve_comparison.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"Saved: {out_path}")
plt.show()
